# Colab: resumable bridge-safe coder adapter training + telemetry

This is a **separate training notebook**. Keep `CGR_Specialist_Adapter_Routing_Bridge.ipynb` as the dataset-free Bridge verification run. This notebook trains a new coder LoRA on the exact `Qwen/Qwen3-0.6B` base, checkpoints to Drive every 25 optimizer steps, resumes from the latest complete checkpoint after a Colab disconnect, and writes step/eval logs and paper-ready metrics to Drive.

The existing public coder adapter targets q/k/v/o. It cannot substantiate the requested frozen-k/v target claim. To measure that design, this run starts a separate adapter with only `q_proj`, `gate_proj`, `up_proj`, and `down_proj` targets. This is a fresh adapter experiment, not a continuation of the existing coder adapter. The train corpus is public Python instruction data; validation loss is a training diagnostic, not evidence of CGR repair success.

**Durable files:** `checkpoint-N/` (adapter + optimizer/scheduler/RNG state), `training_progress.jsonl`, TensorBoard event files, `training_history.json`, `adapter_metadata_summary.json`, and final adapter safetensors—all under a run-specific Google Drive folder. Keep the same `RUN_ID` to resume; change it for a new run.


In [8]:
%pip -q install transformers==4.57.6 "peft>=0.15,<1" "datasets>=3.6,<5" "torchao>=0.16.0" accelerate safetensors huggingface_hub tensorboard

In [4]:
import json, math, os, time, shutil
from datetime import datetime, timezone
from pathlib import Path
import torch, transformers
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling, TrainerCallback)
from transformers.trainer_utils import get_last_checkpoint
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU (T4 or better).')
drive.mount('/content/drive')
RUN_ID = 'coder_qwen3_0.6b_q_mlp_r16_v1'  # keep unchanged when resuming this run
ROOT = Path('/content/drive/MyDrive/cgr_adapter_checkpoints')
OUTPUT_DIR = ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / 'tensorboard'
LOG_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
DATASET_ID = 'iamtarun/python_code_instructions_18k_alpaca'
TARGET_MODULES = ['q_proj', 'gate_proj', 'up_proj', 'down_proj']
MAX_LENGTH = 512
MAX_TRAIN_SAMPLES = None  # e.g. 2000 for a short pipeline check
RESUME_FROM_LATEST = True
SAVE_STEPS = 25
LOGGING_STEPS = 5
EVAL_STEPS = 25

api = HfApi()
BASE_REVISION = api.model_info(BASE_MODEL_ID).sha
DATASET_REVISION = api.dataset_info(DATASET_ID).sha
DEVICE_NAME = torch.cuda.get_device_name(0)
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
SESSION_START_MONOTONIC = time.monotonic()
print('GPU:', DEVICE_NAME, '| Transformers:', transformers.__version__)
print('Base commit:', BASE_REVISION, '| dataset commit:', DATASET_REVISION)
print('Output:', OUTPUT_DIR)

Mounted at /content/drive
GPU: Tesla T4 | Transformers: 4.57.6
Base commit: c1899de289a04d12100db370d81485cdf75e47ca | dataset commit: 7cae181e29701a8663a07a3ea43c8e105b663ba1
Output: /content/drive/MyDrive/cgr_adapter_checkpoints/coder_qwen3_0.6b_q_mlp_r16_v1


## 1. Load and record the exact data split

This uses the dataset associated with the public coder adapter. The split is reproducible and the dataset commit plus row counts are written to the run manifest. No Kaggle account or external secret is needed.


In [5]:
raw = load_dataset(DATASET_ID, revision=DATASET_REVISION)
source = raw['train'] if 'train' in raw else raw[next(iter(raw.keys()))]
splits = source.train_test_split(test_size=0.02, seed=42)
train_raw, eval_raw = splits['train'], splits['test']
if MAX_TRAIN_SAMPLES is not None:
    train_raw = train_raw.select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
print('Rows:', len(train_raw), 'train /', len(eval_raw), 'validation')
print('Fields:', train_raw.column_names)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_row(row):
    instruction = str(row.get('instruction') or row.get('prompt') or row.get('text') or '').strip()
    context = str(row.get('input') or '').strip()
    answer = str(row.get('output') or row.get('response') or row.get('completion') or '').strip()
    if not instruction or not answer:
        return {'text':''}
    user_text = instruction + (f'\n\nInput:\n{context}' if context else '')
    messages = [{'role':'user','content':user_text},{'role':'assistant','content':answer}]
    return {'text':tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_text = train_raw.map(format_row).filter(lambda r: bool(r['text']))
eval_text = eval_raw.map(format_row).filter(lambda r: bool(r['text']))
def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
train_ds = train_text.map(tokenize_batch, batched=True, remove_columns=train_text.column_names)
eval_ds = eval_text.map(tokenize_batch, batched=True, remove_columns=eval_text.column_names)
token_count_train = sum(len(x) for x in train_ds['input_ids'])
print('Tokenized examples:', len(train_ds), '/', len(eval_ds), '| train tokens:', token_count_train)


README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

Rows: 18239 train / 373 validation
Fields: ['instruction', 'input', 'output', 'prompt']


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/373 [00:00<?, ? examples/s]

Filter:   0%|          | 0/373 [00:00<?, ? examples/s]

Map:   0%|          | 0/18239 [00:00<?, ? examples/s]

Map:   0%|          | 0/373 [00:00<?, ? examples/s]

Tokenized examples: 18239 / 373 | train tokens: 2627738


## 2. Build the exact-base, query/MLP-only adapter

The `target_modules` list is recorded and then checked against actual trainable LoRA parameter names. The check fails if any adapter parameter appears under `k_proj` or `v_proj`. This verifies the trained module selection; it does not by itself prove all cache states are invariant when the adapter is active. The Bridge notebook separately tests retained-prefix K/V identity and base-prefill logits.


In [9]:
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, revision=BASE_REVISION, torch_dtype=torch.float16,
    device_map={'':0}, low_cpu_mem_usage=True
)
base.config.use_cache = False
base.gradient_checkpointing_enable()
base.enable_input_require_grads()
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=TARGET_MODULES, task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base, peft_config)
model.print_trainable_parameters()
trainable_params, all_params = model.get_nb_trainable_parameters()
actual_lora_modules = sorted({name.split('.lora_')[0].split('.')[-1]
                              for name, _ in model.named_parameters() if '.lora_' in name})
unexpected = sorted(set(actual_lora_modules) - set(TARGET_MODULES))
if unexpected or {'k_proj','v_proj'} & set(actual_lora_modules):
    raise RuntimeError(f'Unexpected LoRA targets: {unexpected}; actual={actual_lora_modules}')
print('Verified active LoRA modules:', actual_lora_modules)


trainable params: 6,881,280 || all params: 602,931,200 || trainable%: 1.1413
Verified active LoRA modules: ['down_proj', 'gate_proj', 'q_proj', 'up_proj']


## 3. Durable checkpointing, step logs, and resume

Checkpoints are written directly to Drive every 25 optimizer steps and include optimizer, scheduler, and RNG state (`save_only_model=False`). The log callback flushes each record to JSONL as training proceeds, so the last logged state survives even if the Colab runtime disconnects before the next checkpoint. On rerun, keep `RUN_ID` and `RESUME_FROM_LATEST=True`; the notebook loads the latest complete checkpoint.


In [10]:
PROGRESS_JSONL = OUTPUT_DIR / 'training_progress.jsonl'
BASELINE_JSON = OUTPUT_DIR / 'baseline_eval.json'
MANIFEST_JSON = OUTPUT_DIR / 'run_manifest.json'

run_manifest = {
    'run_id':RUN_ID, 'base_model_id':BASE_MODEL_ID, 'base_revision':BASE_REVISION,
    'dataset_id':DATASET_ID, 'dataset_revision':DATASET_REVISION,
    'target_modules_requested':TARGET_MODULES, 'lora_rank':16, 'lora_alpha':32,
    'max_length':MAX_LENGTH, 'train_examples':len(train_ds), 'eval_examples':len(eval_ds),
    'train_tokens':token_count_train, 'transformers_version':transformers.__version__,
    'torch_version':torch.__version__, 'device_name':DEVICE_NAME,
    'run_started_utc':RUN_STARTED_UTC, 'save_steps':SAVE_STEPS,
    'logging_steps':LOGGING_STEPS, 'eval_steps':EVAL_STEPS,
}
if MANIFEST_JSON.exists():
    previous = json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))
    if previous.get('base_revision') != BASE_REVISION or previous.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Run ID already contains a different base or dataset revision; choose a new RUN_ID.')
    run_manifest['run_started_utc'] = previous.get('run_started_utc', RUN_STARTED_UTC)
MANIFEST_JSON.write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

class DriveTelemetryCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.session_start = time.monotonic()
        self.prior_elapsed = 0.0
        if self.log_path.exists():
            for line in self.log_path.read_text(encoding='utf-8').splitlines():
                try:
                    self.prior_elapsed = max(self.prior_elapsed, float(json.loads(line).get('cumulative_elapsed_seconds', 0)))
                except Exception:
                    pass
    def _write(self, record):
        with self.log_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False, default=float) + '\n')
            f.flush(); os.fsync(f.fileno())
    def on_log(self, args, state, control, logs=None, **kwargs):
        elapsed = time.monotonic() - self.session_start
        record = {
            'event':'log', 'logged_utc':datetime.now(timezone.utc).isoformat(),
            'global_step':state.global_step, 'epoch':state.epoch,
            'session_elapsed_seconds':elapsed,
            'cumulative_elapsed_seconds':self.prior_elapsed + elapsed,
            'peak_vram_gb':torch.cuda.max_memory_allocated()/1024**3,
            'metrics':logs or {},
        }
        self._write(record)
    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        self._write({'event':'checkpoint_saved','logged_utc':datetime.now(timezone.utc).isoformat(),
                     'global_step':state.global_step,'checkpoint_path':str(ckpt)})

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), run_name=RUN_ID,
    num_train_epochs=1, per_device_train_batch_size=1, per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, gradient_checkpointing=True, fp16=True,
    learning_rate=2e-5, lr_scheduler_type='cosine', warmup_ratio=0.03,
    eval_strategy='steps', eval_steps=EVAL_STEPS,
    logging_strategy='steps', logging_steps=LOGGING_STEPS, logging_first_step=True,
    save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=3, save_only_model=False,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    logging_dir=str(LOG_DIR), report_to=['tensorboard'],
    include_tokens_per_second=True, include_num_input_tokens_seen=True,
    skip_memory_metrics=False, optim='adamw_torch',
    seed=42, data_seed=42, overwrite_output_dir=False,
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[DriveTelemetryCallback(PROGRESS_JSONL)],
)
latest_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
if RESUME_FROM_LATEST and latest_checkpoint:
    print('Will resume optimizer/model/scheduler/RNG from:', latest_checkpoint)
else:
    print('No resumable checkpoint; starting a new run.')


The model is already on multiple devices. Skipping the move to device specified in `args`.


No resumable checkpoint; starting a new run.


## 4. Capture baseline, train, and save a final adapter snapshot

The baseline validation metrics are saved before training and reused on resume, so resuming does not mistake the freshly initialized adapter for the original baseline. Trainer logs/checkpoint state hold the step-wise loss and learning-rate schedule.


In [ ]:
if latest_checkpoint and BASELINE_JSON.exists():
    initial_eval = json.loads(BASELINE_JSON.read_text(encoding='utf-8'))
else:
    initial_eval = trainer.evaluate()
    initial_eval['eval_perplexity'] = math.exp(min(initial_eval['eval_loss'], 700))
    BASELINE_JSON.write_text(json.dumps(initial_eval, indent=2), encoding='utf-8')

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
training_started = time.monotonic()
try:
    train_result = trainer.train(resume_from_checkpoint=latest_checkpoint if (RESUME_FROM_LATEST and latest_checkpoint) else None)
except Exception as exc:
    failure_record = {'event':'training_exception','utc':datetime.now(timezone.utc).isoformat(),
                      'error':repr(exc),'latest_checkpoint':get_last_checkpoint(str(OUTPUT_DIR))}
    (OUTPUT_DIR/'last_failure.json').write_text(json.dumps(failure_record, indent=2), encoding='utf-8')
    raise
session_wall_seconds = time.monotonic() - training_started

# Final adapter is a compact inference artifact; checkpoint-N folders retain resumable optimizer state.
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
final_eval = trainer.evaluate()
final_eval['eval_perplexity'] = math.exp(min(final_eval['eval_loss'], 700))
final_eval['eval_loss_delta_from_baseline'] = final_eval['eval_loss'] - initial_eval['eval_loss']

# Atomically save complete run telemetry after train() returns.


Step,Training Loss,Validation Loss,Model Preparation Time,Input Tokens Seen
25,1.861300,1.941758,0.149900,28287
50,1.485900,1.541037,0.149900,56258


In [ ]:
history = trainer.state.log_history
training_logs = [x for x in history if 'loss' in x and 'eval_loss' not in x]
eval_logs = [x for x in history if 'eval_loss' in x]
initial_train_loss = training_logs[0]['loss'] if training_logs else None
final_train_loss = training_logs[-1]['loss'] if training_logs else None

def directory_size_bytes(path):
    path=Path(path)
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) if path.exists() else 0

checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
checkpoint_bytes = directory_size_bytes(checkpoint) if checkpoint else 0
final_adapter_bytes = directory_size_bytes(FINAL_ADAPTER_DIR)
trainable_params, total_params = model.get_nb_trainable_parameters()
param_summary = {
    'trainable_parameters':int(trainable_params), 'total_parameters':int(total_params),
    'trainable_percent':100.0*trainable_params/total_params,
    'target_modules_requested':TARGET_MODULES, 'target_modules_observed':actual_lora_modules,
    'k_proj_or_v_proj_adapted':bool({'k_proj','v_proj'} & set(actual_lora_modules)),
}
logs_for_curves = {
    'initial_eval':initial_eval, 'final_eval':final_eval,
    'trainer_log_history':history,
    'drive_progress_jsonl':str(PROGRESS_JSONL),
}
(OUTPUT_DIR/'training_history.json').write_text(json.dumps(logs_for_curves, indent=2, default=float), encoding='utf-8')

hardware = {
    'device_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'peak_vram_gb':torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.0,
    'session_wall_clock_seconds':session_wall_seconds,
    'cumulative_logged_wall_clock_seconds':max([0.0]+[
        float(json.loads(line).get('cumulative_elapsed_seconds',0))
        for line in PROGRESS_JSONL.read_text(encoding='utf-8').splitlines() if line.strip() and json.loads(line).get('event')=='log'
    ]),
    'train_samples_per_second':train_result.metrics.get('train_samples_per_second'),
    'train_tokens_per_second':train_result.metrics.get('train_tokens_per_second'),
}
summary = {
    'run_id':RUN_ID, 'base_model_id':BASE_MODEL_ID, 'base_revision':BASE_REVISION,
    'dataset_id':DATASET_ID, 'dataset_revision':DATASET_REVISION,
    'initial_logged_train_loss':initial_train_loss, 'final_logged_train_loss':final_train_loss,
    'initial_eval_loss':initial_eval.get('eval_loss'), 'initial_eval_perplexity':initial_eval.get('eval_perplexity'),
    'final_eval_loss':final_eval.get('eval_loss'), 'final_eval_perplexity':final_eval.get('eval_perplexity'),
    'best_eval_loss':trainer.state.best_metric, 'best_checkpoint':trainer.state.best_model_checkpoint,
    'global_step':trainer.state.global_step, 'learning_rate_last_logged':next((x.get('learning_rate') for x in reversed(history) if 'learning_rate' in x),None),
    'parameters':param_summary, 'hardware_compute':hardware,
    'latest_checkpoint_path':checkpoint, 'latest_checkpoint_size_mb':checkpoint_bytes/1024**2,
    'final_adapter_path':str(FINAL_ADAPTER_DIR), 'final_adapter_size_mb':final_adapter_bytes/1024**2,
    'saved_utc':datetime.now(timezone.utc).isoformat(),
}
summary_path=OUTPUT_DIR/'adapter_metadata_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, default=float), encoding='utf-8')
print(json.dumps(summary, indent=2, default=float))
print('All run artifacts saved under:',OUTPUT_DIR)


## 5. Inspect logs and resume after interruption

After a disconnect, reconnect Colab, mount the same Drive, keep the same `RUN_ID`, and rerun from the top. The code finds the latest `checkpoint-N` and passes it to `trainer.train(resume_from_checkpoint=...)`. The trainer restores adapter/model weights, optimizer, scheduler, RNG, and global step. Loss/eval events and checkpoint events are append-only in `training_progress.jsonl`; TensorBoard event files and `training_history.json` are also on Drive.

View TensorBoard from a cell with `%load_ext tensorboard` then `%tensorboard --logdir /content/drive/MyDrive/cgr_adapter_checkpoints/coder_qwen3_0.6b_q_mlp_r16_v1/tensorboard`. The exact adapter checkpoint and the larger resumable training checkpoint are different artifacts; use `final_adapter/` for inference and `checkpoint-N/` to resume training.

The notebook records measured numbers. Do not pre-fill the paper's parameter count, adapter size, throughput, or `<4 GB` claim from estimates; read `adapter_metadata_summary.json` after the run and report the measured GPU/runtime. The training notebook does not prove crop bit-identity or bridge coherence; use the separate Bridge verification notebook for those tests.
